# Étape 7 - Synthèse de modélisation et fiche modèle

## Objectif

La modélisation est faite : l'étape 4 a comparé six modèles à une référence naïve,
l'étape 5 a optimisé la forêt, l'étape 5b a choisi entre forêt et boosting, l'étape 5c a
diagnostiqué le modèle retenu puis l'a évalué une seule fois sur le jeu de test. **Ce notebook ne réentraîne
rien**, et c'est délibéré : réentraîner ici créerait un second modèle, donc une seconde
vérité, pour un projet qui n'en veut qu'une.

Il fait ce qui manquait entre le modèle et son usage :

1. **Vérifier la conformité** du modèle livré, par des tests qui échouent si elle est
   rompue, plutôt que par une affirmation dans un document.
2. **Livrer le code d'inférence** : un tiers doit pouvoir prédire sans rejouer un notebook.
3. **Produire la fiche modèle**, qui dit ce que le modèle fait, ce qu'il ne fait pas, et
   pour qui il peut être dangereux.

Protocole et résultats détaillés : [`étape_4.md`](../etapes/étape_4.md),
[`étape_5b.md`](../etapes/étape_5b.md) et [`étape_5c.md`](../etapes/étape_5c.md). La révision
du protocole demandée par la formatrice est décrite dans
[`étape_10_revision.md`](../etapes/étape_10_revision.md).

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_PATH = Path.cwd().parent
sys.path.insert(0, str(PROJECT_PATH))

from modeles.inference import charger, predire, preparer_ligne_brute  # noqa: E402
from modeles.protocole import (CODE_SISE, COLONNES_ENTREE, decouper,  # noqa: E402
                               identifiant_formation)

DATA_PATH = PROJECT_PATH / 'csv' / 'dataset_phase3_final.csv'
CARTE_PATH = PROJECT_PATH / 'MODEL_CARD.md'

modele = charger()
df = pd.read_csv(DATA_PATH, dtype={CODE_SISE: str})

pipeline = modele['pipeline']
estimateur = pipeline.named_steps['modele']

print(f'Modèle : {modele["modele"]} ({type(estimateur).__name__})')
print(f'Variables du modèle : {len(modele["features"])}, '
      f'colonnes reçues par la pipeline : {len(modele["colonnes_entree"])}')
print(f'Cible : {modele["cible"]}')
print(f'Métriques de test (relevées à l\'étape 5c) : {modele["metriques_test"]}')

Modèle : Gradient boosting (HistGradientBoostingRegressor)
Variables du modèle : 18, colonnes reçues par la pipeline : 17
Cible : 6-Taux d'emploi salarié en France - 6 mois après le diplôme
Métriques de test (relevées à l'étape 5c) : {'MAE': 10.09, 'RMSE': 13.081, 'R²': 0.484}


---

## 1. Le protocole, en une page

| Décision | Ce qui a été fait | Où |
|---|---|---|
| Périmètre | 17 065 formations, 18 variables explicatives, 8 catégorielles et 10 numériques | Étape 3 |
| Découpage | 80 / 20 **groupé par formation** (UAI × SISE), `random_state=42` : aucune formation des deux côtés | Étape 3 |
| Prétraitement | Agrégats de contexte, one-hot et passthrough, **tous dans la pipeline** | Étapes 3 et 4 |
| Référence | `DummyRegressor(strategy='mean')`, évaluée dans les mêmes conditions | Étape 4 |
| Comparaison | 6 modèles, validation croisée **groupée** à 5 plis, mêmes plis pour tous | Étape 4 |
| Optimisation | `GridSearchCV` en plis groupés, puis **règle à un écart-type**, pour la forêt et le boosting | Étapes 5 et 5b |
| Sélection | Gradient boosting, vainqueur du duel sur une règle fixée avant le calcul | Étape 5b |
| Diagnostic | Résidus, importance par permutation, erreurs par groupe, validation temporelle, **hors pli** | Étape 5c |
| Évaluation | Jeu de test ouvert **une seule fois** : MAE 10,090, RMSE 13,081, R² 0,484 | Étape 5c |

**Le choix qui structure tout le reste** : la métrique principale est la MAE, en points de
taux d'emploi. Une erreur de 10 points se comprend sans traduction par un responsable de
formation, ce qui n'est pas le cas d'un R carré.

---

## 2. Contrôles de conformité

Les trois règles éliminatoires de la grille sont vérifiables sur le modèle livré, pas
seulement sur les notebooks. Ces contrôles portent sur ce qui a été **sauvegardé** et sur
le code qui l'a produit : si un jour quelqu'un réentraîne en introduisant une fuite, ils
échouent.

In [2]:
MOTIFS_FUITE = ['12-', '18-', '24-', '30-', 'emploi stable', 'emploi non salarié']
IDENTIFIANTS = ["Code UAI de l'établissement", 'Code du diplôme SISE',
                'Établissement', 'Libellé du diplôme']

controles = []


def controler(intitule, condition, detail):
    """Enregistre un contrôle et échoue immédiatement s'il n'est pas satisfait."""
    controles.append({'contrôle': intitule, 'résultat': 'OK' if condition else 'ÉCHEC',
                      'détail': detail})
    assert condition, f'{intitule} : {detail}'


features = modele['features']
colonnes_modele = [nom.split('__', 1)[-1] for nom
                   in pipeline.named_steps['pretraitement'].get_feature_names_out()]

controler('La cible est absente des entrées',
          modele['cible'] not in modele['colonnes_entree'],
          f'{len(modele["colonnes_entree"])} colonnes reçues, aucune n\'est la cible')

fuitantes = [f for f in modele['colonnes_entree'] if any(m in f for m in MOTIFS_FUITE)]
controler('Aucune variable postérieure à la cible',
          not fuitantes,
          'aucune variable mesurée à 12, 18, 24, 30 mois ni sur l\'emploi stable')

controler('Aucun identifiant ne parvient au modèle',
          not [c for c in colonnes_modele if any(c.startswith(i) for i in IDENTIFIANTS)],
          'les codes UAI et SISE ne servent qu\'aux agrégats, le modèle ne les voit pas')

controler('Tout ce qui apprend sur les données est dans la pipeline',
          [nom for nom, _ in pipeline.steps] == ['agregats', 'pretraitement', 'modele'],
          'agrégats, encodage et modèle, réajustés sur le seul train à chaque pli')

controler('Les agrégats sont embarqués dans la pipeline',
          set(pipeline.named_steps['agregats'].defauts_) == {
              'taille_mediane_secteur', 'nb_formations_etablissement',
              'nb_etablissements_par_diplome'},
          'tables et valeurs de repli apprises sur l\'apprentissage, sans référentiel séparé')

controler('Le hasard est fixé',
          estimateur.random_state == 42,
          f'random_state={estimateur.random_state} sur le modèle')

index_train, index_test = decouper(df)
formations = identifiant_formation(df)
communes = set(formations[index_train]) & set(formations[index_test])
controler('Aucune formation à la fois en apprentissage et en test',
          not communes,
          f'{formations[index_train].nunique()} formations d\'un côté, '
          f'{formations[index_test].nunique()} de l\'autre, aucune en commun')

# Pas d'écart entre entraînement et inférence : une ligne brute passée par le module
# d'inférence reçoit la même prédiction que la pipeline sur la même ligne.
echantillon = df.loc[index_train].head(200)
via_inference = pd.concat([preparer_ligne_brute(ligne, modele)
                           for ligne in echantillon.to_dict('records')], ignore_index=True)
premier = pipeline.predict(via_inference)
second = pipeline.predict(echantillon[COLONNES_ENTREE])
ecart_max = float(np.abs(premier - second).max())
controler('Inférence identique à l\'entraînement',
          np.allclose(premier, second, rtol=1e-10, atol=1e-10),
          f'écart maximal de {ecart_max:.1e} point sur 200 lignes brutes')

controler('Les métriques de test sont livrées avec le modèle',
          set(modele['metriques_test']) == {'MAE', 'RMSE', 'R²'},
          'MAE, RMSE et R² voyagent avec le modèle, pas seulement dans un rapport')

# Le test ne s'ouvre qu'une fois : X_test et y_test ne doivent apparaître que dans les
# cellules qui construisent et contrôlent le découpage, à l'étape 3, et dans l'unique
# cellule d'évaluation de l'étape 5c.
usages = []
for chemin in sorted((PROJECT_PATH / 'notebooks').glob('etape_*.ipynb')):
    if chemin.stem.startswith('etape_7'):
        continue
    carnet = json.loads(chemin.read_text(encoding='utf-8'))
    for numero, cellule in enumerate(carnet['cells']):
        source = ''.join(cellule['source'])
        if cellule['cell_type'] == 'code' and ('X_test' in source or 'y_test' in source):
            usages.append(f'{chemin.stem}[{numero}]')
hors_decoupage = [u for u in usages if not u.startswith('etape_3')]
controler('Le jeu de test n\'est lu que dans une cellule de mesure',
          len(hors_decoupage) == 1 and hors_decoupage[0].startswith('etape_5c'),
          f'cellules qui le touchent : {", ".join(usages)}')

print(f'{len(controles)} contrôles exécutés, tous passés.')
pd.DataFrame(controles).set_index('contrôle')

10 contrôles exécutés, tous passés.


,résultat,détail
contrôle,,
La cible est absente des entrées,OK,"17 colonnes reçues, aucune n'est la cible"
Aucune variable postérieure à la cible,OK,"aucune variable mesurée à 12, 18, 24, 30 mois ..."
Aucun identifiant ne parvient au modèle,OK,les codes UAI et SISE ne servent qu'aux agréga...
Tout ce qui apprend sur les données est dans la pipeline,OK,"agrégats, encodage et modèle, réajustés sur le..."
Les agrégats sont embarqués dans la pipeline,OK,tables et valeurs de repli apprises sur l'appr...
Le hasard est fixé,OK,random_state=42 sur le modèle
Aucune formation à la fois en apprentissage et en test,OK,"3958 formations d'un côté, 990 de l'autre, auc..."
Inférence identique à l'entraînement,OK,écart maximal de 0.0e+00 point sur 200 lignes ...
Les métriques de test sont livrées avec le modèle,OK,"MAE, RMSE et R² voyagent avec le modèle, pas s..."


**Ce que ces contrôles garantissent.**

Ils garantissent que le fichier livré est conforme : pas de cible ni de variable
postérieure dans les entrées, aucun identifiant transmis au modèle, des agrégats appris dans
la pipeline et non à côté, un hasard fixé. Ils vérifient aussi deux propriétés que la
première version du projet n'avait pas, et que la relecture de la formatrice a demandées :

- **aucune formation n'est à la fois en apprentissage et en test** : le score de test porte
  sur des formations jamais vues ;
- **l'inférence reproduit exactement l'entraînement** : il n'y a plus de référentiel calculé
  à part, la pipeline porte elle-même ses tables d'agrégats.

**La règle « le test s'ouvre une fois » devient vérifiable.** Le dernier contrôle lit le
code des notebooks : `X_test` et `y_test` n'apparaissent que dans la cellule qui découpe, à
l'étape 3, et dans la cellule qui mesure, à l'étape 5c. Toute analyse qui réutiliserait le
test ferait échouer ce notebook.

---

## 3. Le modèle est-il utilisable par un tiers ?

Un modèle qui ne s'utilise qu'en rejouant le notebook qui l'a produit n'est pas livré. Le
module [`modeles/inference.py`](../modeles/inference.py) prend une formation telle
qu'InserSup la publie, reconstruit les six variables ligne à ligne avec les définitions de
[`modeles/protocole.py`](../modeles/protocole.py), et laisse la pipeline calculer les trois
agrégats avec les tables apprises sur l'apprentissage.

In [3]:
CAS_REFERENCE = {
    'Région': 'Bretagne', 'Académie': 'Rennes',
    'Type de diplôme': 'Licence professionnelle',
    'Domaine disciplinaire': 'Sciences, technologies, santé',
    'Discipline': 'Sciences fondamentales et applications',
    'Secteur disciplinaire': 'Informatique', 'Promotion': 2024,
    "Code UAI de l'établissement": '0350936C', 'Code du diplôme SISE': '99999',
    '6-Nombre de sortants - 6 mois après le diplôme': 48,
    '6-Nombre de poursuivants - 6 mois après le diplôme': 6,
}

# Non-régression : le module doit reproduire exactement la démonstration de l'étape 5c.
reference = predire(modele, CAS_REFERENCE)
assert abs(reference['prediction'] - 63.3) < 0.05, (
    f"Le module d'inférence diverge de l'étape 5c : {reference['prediction']} au lieu de 63.3")
print(f"Non-régression vérifiée : {reference['prediction']} %, comme à l'étape 5c")
print(f"Variables retombées sur un défaut : {', '.join(reference['replis'])}")
print()

# Trois cas contrastés, pour voir ce que le modèle fait varier.
variantes = {
    'Licence pro informatique (référence)': CAS_REFERENCE,
    'Même formation, en licence générale': {
        **CAS_REFERENCE, 'Type de diplôme': 'Licence générale'},
    'Master LMD en langues, promotion 2020': {
        **CAS_REFERENCE, 'Type de diplôme': 'Master LMD', 'Promotion': 2020,
        'Domaine disciplinaire': 'Lettres, langues et arts',
        'Discipline': 'Langues et littératures étrangères',
        'Secteur disciplinaire': 'Langues et littératures étrangères'},
}

resultats = []
for intitule, cas in variantes.items():
    sortie = predire(modele, cas)
    resultats.append({
        'cas': intitule,
        'prédiction': f"{sortie['prediction']} %",
        'intervalle': f"{sortie['intervalle'][0]} à {sortie['intervalle'][1]} %",
        'valeurs par défaut': len(sortie['replis']),
    })

pd.DataFrame(resultats).set_index('cas')

Non-régression vérifiée : 63.3 %, comme à l'étape 5c
Variables retombées sur un défaut : nb_formations_etablissement, nb_etablissements_par_diplome



,prédiction,intervalle,valeurs par défaut
cas,,,
Licence pro informatique (référence),63.3 %,53.2 à 73.4 %,2
"Même formation, en licence générale",56.9 %,46.8 à 67.0 %,2
"Master LMD en langues, promotion 2020",36.1 %,26.0 à 46.2 %,2


**Lecture.** Les trois cas ne diffèrent que par ce qu'on fait varier, et l'écart est celui
qu'annonçait l'analyse : passer d'une licence professionnelle à une licence générale, à
effectif, région et promotion identiques, fait chuter la prédiction. Le troisième cas cumule
les facteurs défavorables identifiés à l'étape 2, discipline littéraire et promotion 2020,
et se retrouve nettement plus bas.

**Une prédiction ne se lit jamais seule.** L'intervalle affiché vaut plus ou moins la MAE de
test, soit environ 10,1 points. Il est indicatif, pas statistique : ce n'est pas un
intervalle de confiance, c'est l'ordre de grandeur de l'erreur habituelle. Sur une formation
de 25 sortants ou moins, il faudrait le lire plus large encore : l'étape 5c y mesure
11,75 points de MAE.

**Les modalités inconnues ne font pas échouer la prédiction.** Ni l'établissement
`0350936C` ni le diplôme `99999` ne figurent dans l'apprentissage : les deux agrégats qui en
dépendent retombent sur leur médiane, et la fonction le signale plutôt que de le taire.
C'est la situation normale d'une formation nouvelle, celle que le score de test mesure.

In [4]:
# Relevés de l'étape 5c, lus dans le modèle livré : rien n'est recopié à la main, et ce
# notebook n'ouvre pas le jeu de test.
metriques = modele['metriques_test']
resume = modele['resume_test']
releves = modele['releves_validation']
parametres = ', '.join(f'`{cle}={valeur}`' for cle, valeur in estimateur.get_params().items()
                       if cle in ('learning_rate', 'max_iter', 'max_leaf_nodes',
                                  'min_samples_leaf', 'n_estimators', 'max_features',
                                  'random_state'))
gain = resume['mae_reference'] - metriques['MAE']

carte = f"""# Fiche du modèle : insertion professionnelle des formations

*Fiche générée par [`etape_7_synthese_modele.ipynb`](notebooks/etape_7_synthese_modele.ipynb)
à partir du modèle livré. Ne pas modifier à la main.*

## Ce que fait ce modèle

Il prédit le **taux d'emploi salarié en France six mois après le diplôme** d'une formation
de l'enseignement supérieur, à partir de {len(modele['features'])} caractéristiques.

- **Tâche** : régression, cible continue de 0 à 100 %.
- **Algorithme** : {modele['modele']} (`{type(estimateur).__name__}`), {parametres}.
- **Unité de prédiction** : une formation, c'est-à-dire un diplôme d'un établissement pour
  une promotion donnée. **Jamais une personne.**

## Données d'entraînement

- **Source** : InserSup, millésime 2026_S1, ministère de l'Enseignement supérieur, Licence
  Ouverte Etalab.
- **Périmètre** : 17 065 lignes, promotions 2019 à 2024, France.
- **Découpage** : 80 % des formations en apprentissage, 20 % en test, **groupé par
  formation** (code UAI × code SISE), `random_state=42`. Aucune formation n'a de promotion
  des deux côtés.
- **Pipeline** : agrégats de contexte appris sur l'apprentissage, encodage one-hot, modèle.
  Rien de ce qui apprend sur les données n'est calculé hors de la pipeline.
- **Exclusions volontaires** : tous les indicateurs d'emploi mesurés à 12, 18, 24 et
  30 mois, ainsi que l'emploi stable et non salarié à 6 mois. Mesurés au même horizon que la
  cible ou après elle, ils constitueraient une fuite de données.

## Performance

| Métrique | Valeur | Lecture |
|---|---|---|
| MAE | **{metriques['MAE']} points** | Erreur moyenne sur {resume['lignes']} lignes, {resume['formations']} formations jamais vues |
| RMSE | {metriques['RMSE']} points | Pénalise les grosses erreurs |
| R² | {metriques['R²']} | Part de la variance expliquée |
| Référence naïve | {resume['mae_reference']} points de MAE | Prédire la moyenne pour tout le monde |
| **Gain** | **{gain:.2f} points, soit {100 * resume['reduction_erreur']:.0f} %** | Ce que le modèle apporte réellement |

Le jeu de test n'a été ouvert qu'une fois, après le choix définitif du modèle. Son score
rejoint celui de la validation croisée groupée ({modele['metriques_validation']['MAE']} de
MAE) : le protocole n'est pas optimiste.

## Où ce modèle échoue

Mesures en validation croisée groupée, sur des prédictions hors pli.

| Situation | Effet mesuré |
|---|---|
| Formations de 25 sortants ou moins | MAE de {releves['mae_petites_formations']:.2f} points, contre {releves['mae_grandes_formations']:.2f} au-delà de 90 sortants |
| Taux extrêmes | Sous-estimation de {releves['residu_taux_hauts']:.0f} points en moyenne au-delà de 85 %, surestimation de {-releves['residu_taux_bas']:.0f} points sous 25 % |
| Prédire une promotion future | R² de {releves['r2_temporel']:.3f} en entraînant sur 2019-2022 et en évaluant sur 2023-2024 |
| Choc conjoncturel inédit | Promotion 2020 la plus mal prédite, MAE de {releves['mae_promotion_2020']:.2f} points |

## Limites d'usage

1. **Le modèle décrit, il n'explique pas.** Une variable importante n'est pas une cause. Le
   caractère professionnalisant du diplôme prédit bien parce qu'il résume un public, une
   discipline et un marché.
2. **Variables contemporaines de la cible.** Les effectifs de sortants et de poursuivants
   proviennent de la même enquête à six mois : le modèle prédit au moment de l'enquête, pas
   avant la sortie de promotion.
3. **Seuil de publication.** Aucun taux n'est publié sous 20 sortants : les petites
   formations sont absentes des données, et donc hors du domaine de validité.
4. **Un taux d'emploi salarié n'est pas un taux d'insertion.** L'emploi non salarié est
   mesuré séparément et exclu.

## Ce pour quoi ce modèle ne doit pas être utilisé

- **Décider du financement ou de la fermeture d'une formation.** L'erreur moyenne avoisine
  10 points et monte nettement sur les petites formations : elle est du même ordre que les
  écarts qu'on voudrait arbitrer.
- **Comparer deux formations proches.** Un écart de 5 points prédits est dans le bruit.
- **Évaluer une personne.** L'unité de prédiction est une formation entière.
- **Classer des établissements.** Un établissement spécialisé hérite de la performance de
  son secteur, l'étape 2 le montre.

## Biais connus

- **Biais de sélection** : les formations sous 20 sortants sont exclues par la source. Les
  petites formations, souvent rurales ou spécialisées, ne sont pas représentées.
- **Biais de représentation** : 58 % des lignes du fichier brut n'ont pas de cible et
  disparaissent du périmètre ; les diplômes visés des écoles privées sont les plus mal
  prédits.
- **Biais d'interprétation** : le taux d'emploi salarié pénalise mécaniquement les formations
  qui mènent à une poursuite d'études. Un taux bas n'y signale aucun échec.
- **Angle mort** : genre, nationalité et régime d'inscription ne sont pas dans le modèle. Il
  ne peut rien dire des disparités, et ne doit pas être invoqué pour les nier.

## Utilisation

```python
from modeles.inference import charger, predire

modele = charger()
predire(modele, {{
    'Région': 'Bretagne',
    'Académie': 'Rennes',
    'Type de diplôme': 'Licence professionnelle',
    'Domaine disciplinaire': 'Sciences, technologies, santé',
    'Discipline': 'Sciences fondamentales et applications',
    'Secteur disciplinaire': 'Informatique',
    'Promotion': 2024,
    "Code UAI de l'établissement": '0350936C',
    'Code du diplôme SISE': '99999',
    '6-Nombre de sortants - 6 mois après le diplôme': 48,
    '6-Nombre de poursuivants - 6 mois après le diplôme': 6,
}})
```

Un établissement ou un diplôme inconnu ne fait pas échouer la prédiction : l'agrégat
concerné retombe sur la médiane d'apprentissage, et la clé `replis` de la réponse dit
lesquels.

## Maintenance

Le modèle est lié au millésime 2026_S1. À chaque nouveau millésime InserSup, réexécuter les
notebooks des étapes 1 à 5c dans l'ordre, puis ce notebook : les contrôles de conformité
échouent si une fuite, un identifiant ou une réutilisation du test s'est glissé dans le
projet.
"""

CARTE_PATH.write_text(carte, encoding='utf-8')
print(f'Fiche écrite : {CARTE_PATH.relative_to(PROJECT_PATH)} '
      f'({len(carte.splitlines())} lignes)')
print()
print('\n'.join(carte.splitlines()[:18]))

Fiche écrite : MODEL_CARD.md (120 lignes)

# Fiche du modèle : insertion professionnelle des formations

*Fiche générée par [`etape_7_synthese_modele.ipynb`](notebooks/etape_7_synthese_modele.ipynb)
à partir du modèle livré. Ne pas modifier à la main.*

## Ce que fait ce modèle

Il prédit le **taux d'emploi salarié en France six mois après le diplôme** d'une formation
de l'enseignement supérieur, à partir de 18 caractéristiques.

- **Tâche** : régression, cible continue de 0 à 100 %.
- **Algorithme** : Gradient boosting (`HistGradientBoostingRegressor`), `learning_rate=0.03`, `max_features=1.0`, `max_iter=400`, `max_leaf_nodes=15`, `min_samples_leaf=20`, `random_state=42`.
- **Unité de prédiction** : une formation, c'est-à-dire un diplôme d'un établissement pour
  une promotion donnée. **Jamais une personne.**

## Données d'entraînement

- **Source** : InserSup, millésime 2026_S1, ministère de l'Enseignement supérieur, Licence


---

## Checklist de la phase 7

**7.1 Rigueur méthodologique**

- [x] Découpage groupé par formation réalisé avant tout ce qui apprend sur les données
- [x] `stratify` sans objet : régression
- [x] Tout le prétraitement dans un `ColumnTransformer` / `Pipeline`
- [x] Aucune colonne fuitante dans `X`, identifiants exclus, vérifié par `assert` ci-dessus
- [x] Le jeu de test n'a servi qu'une fois, à l'étape 5c, vérifié en lisant le code des notebooks
- [x] `random_state` fixé partout, et contrôlé sur le modèle livré

**7.2 Baseline et comparaison**

- [x] `DummyRegressor` calculé dans les mêmes conditions que les modèles
- [x] 6 modèles comparés, dont la régression linéaire et Ridge
- [x] `GridSearchCV` avec 5 plis groupés, tableaux `cv_results_` commentés aux étapes 5 et 5b
- [x] Choix final par une règle fixée avant le calcul, réglages par la règle à un écart-type

**7.3 Évaluation**

- [x] Métrique principale justifiée par le métier : la MAE, en points de taux d'emploi
- [x] MAE, RMSE dans l'unité de la cible, R² reporté, résidus analysés
- [x] Écart entraînement / validation traduit en nombre de formations
- [x] Score de test reporté honnêtement, gain sur la référence naïve chiffré

**7.4 Interprétation et limites**

- [x] Importance par permutation en validation croisée groupée, moyennée sur 5 plis
- [x] Cohérence avec l'EDA discutée : une hypothèse infirmée, une nuancée
- [x] Cas d'échec identifiés et mesurés : petites formations, taux extrêmes, 2020, prévision temporelle
- [x] Biais de sélection, de représentation et d'interprétation discutés dans la fiche

## Utilisation de l'IA sur cette étape

| Prompt utilisé | Ce que l'IA a produit | Vérification effectuée |
|---|---|---|
| « Que contient une bonne fiche modèle ? » | Structure en sections, dont « usage prévu » et « hors périmètre » | Retenue, mais la section la plus utile, « ce pour quoi ce modèle ne doit pas être utilisé », a été écrite à partir des erreurs mesurées à l'étape 5, pas d'un modèle générique |
| « Écris des tests pour vérifier l'absence de fuite dans un modèle sauvegardé » | Boucle sur les noms de colonnes avec motifs interdits | Conservée, mais complétée : un test de déterminisme et un contrôle du `random_state`, que la proposition initiale ne couvrait pas |
| « Mon modèle peut-il servir à décider de fermer une formation ? » | Réponse nuancée sur les usages à haut risque | Traduite en critère chiffré : l'erreur moyenne est du même ordre que les écarts à arbitrer, donc non |

Le troisième échange est celui qui a le plus changé le livrable : il a transformé une
intuition, « attention à l'usage », en une limite argumentée par les chiffres du projet.